# PlantDoctor - train the multi-species model (Tomato + Apple + Cherry + Peach)

Extends the 10-class tomato-only model to 18 classes across four species, using the exact
same pipeline as `train_tomato_model_kaggle.ipynb` (same dataset, same fixes, same local
conversion steps) - only the class list changes, since PlantVillage stores every species'
classes as sibling folders under the same `color/` directory. No new data source, no new
app code, no new architecture.

1. **Settings (right sidebar) > Accelerator > GPU T4 x2** (or P100).
2. **Add Data** (right sidebar) > search `plantvillage dataset` > add the same **"PlantVillage Dataset"** by `abdallahalidev` used for the tomato model.
3. Run all cells top to bottom - training happens here on Kaggle's GPU, then you download a small `model_export.zip` and convert it to TF.js **locally**, same as the tomato model (see that notebook's end, or `CLAUDE.md`, for the local conversion steps and why they're local).

## 1. Data

PlantVillage's `Tomato___*`, `Apple___*`, `Cherry_(including_sour)___*`, and `Peach___*` folders
all live as siblings under the same `color/` directory - so "training on 4 species" is just
widening the class whitelist passed to `image_dataset_from_directory`, not merging separate
datasets. Expect **18 classes**: 10 tomato (healthy + 9 diseases), 4 apple (healthy + scab +
black rot + cedar apple rust), 2 cherry (healthy + powdery mildew), 2 peach (healthy +
bacterial spot).

In [ ]:
import os, pathlib

SPECIES = ["Tomato", "Apple", "Cherry", "Peach"]

def matches_species(dirname):
    return "___" in dirname and any(dirname.startswith(s) for s in SPECIES)

# os.walk + pruning any "Crop___Disease"-shaped folder (regardless of crop)
# keeps this to just the directory skeleton instead of listing every one of
# the ~50k+ image files in the full 38-class dataset - see the tomato
# notebook for why a plain rglob was slow here.
candidate_dirs = set()
for root, dirnames, _files in os.walk("/kaggle/input"):
    keep = []
    for d in dirnames:
        if "___" in d:
            if matches_species(d):
                candidate_dirs.add(pathlib.Path(root))
            continue  # leaf class folder full of images - don't descend into it
        keep.append(d)
    dirnames[:] = keep

candidates = sorted(candidate_dirs)
assert candidates, "No matching class folders found - did you add the PlantVillage dataset via 'Add Input'?"

color_dirs = [p for p in candidates if p.name == "color"]
DATA_DIR = color_dirs[0] if color_dirs else candidates[0]
print("Using:", DATA_DIR)

In [ ]:
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir() and matches_species(d.name))
print(len(classes), "classes:")
for c in classes:
    n = len(list((DATA_DIR / c).glob("*.JPG"))) + len(list((DATA_DIR / c).glob("*.jpg")))
    print(f"  {c}: {n} images")

You should see exactly 18 classes. If the count looks off, double check the dataset was added
via 'Add Input' and that `SPECIES` above matches the folder-name prefixes you expect.

## 2. Build train/val datasets

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_names=classes)

class_names = train_ds.class_names  # authoritative order used for the labels array
print(class_names)

# Normalize to [-1, 1] here in the data pipeline, matching exactly what
# App.js's classifyImage does (.div(127.5).sub(1)) before calling the model.
# The exported model must NOT also normalize internally - see CLAUDE.md /
# the tomato notebook for the double-normalization bug this avoids.
def normalize(x, y):
    return x / 127.5 - 1.0, y

train_ds = train_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(normalize).prefetch(tf.data.AUTOTUNE)
# No .cache(): caching decoded 224x224 images for 4 species in RAM risks the
# same out-of-memory kernel crash seen training on tomato alone - disk read +
# decode is cheap next to the GPU compute anyway.

## 3. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

# Named separately (not inline) so the export cell later can reuse these
# exact trained layer objects to build a clean inference-only graph.
pooling = tf.keras.layers.GlobalAveragePooling2D()
dropout = tf.keras.layers.Dropout(0.2)
classifier = tf.keras.layers.Dense(len(class_names), activation="softmax")

# Training graph only - includes augmentation (helps generalization, and is
# a no-op at inference time anyway). Input here is already normalized to
# [-1, 1] by the tf.data pipeline above, matching what App.js hands the
# model, so no preprocessing layer is needed in the graph.
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = pooling(x)
x = dropout(x)
outputs = classifier(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train (frozen base, fast)

More classes and more images than the tomato-only run, so this takes somewhat longer - still
well within a single Kaggle GPU session.

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=8)

## 5. Optional: fine-tune the top of MobileNetV2

Usually improves accuracy a few points. Skip this cell if step 4's validation accuracy is already
good enough.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 6. Evaluate

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Validation accuracy: {acc:.3f}")

## 7. Save the trained model

Same reasoning as the tomato notebook: converting to TF.js happens locally, not here - Kaggle's
pre-installed `tensorflow`/`tf_keras`/`tensorflow_decision_forests` versions fight the
`tensorflowjs` pip package no matter how it's installed on Kaggle. Build a clean inference-only
model (no augmentation, no in-graph preprocessing) reusing the trained layer objects, and save
just that.

In [ ]:
inference_inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input")
x = base_model(inference_inputs, training=False)
x = pooling(x)
x = dropout(x, training=False)
outputs = classifier(x)
inference_model = tf.keras.Model(inference_inputs, outputs)

inference_model.save("/kaggle/working/model.h5")

import json
metadata = {
    "modelName": "plantdoctor-multi-species-18class",
    "labels": class_names,
    "imageSize": IMG_SIZE,
}
with open("/kaggle/working/metadata.json", "w") as f:
    json.dump(metadata, f)

print("Saved:")
!ls -la /kaggle/working/model.h5 /kaggle/working/metadata.json

In [ ]:
!cd /kaggle/working && zip -q model_export.zip model.h5 metadata.json
print("Done - open the notebook's Output pane (after Save Version) and download model_export.zip")

## 8. Download

**Save Version > Save & Run All (Commit)**, wait for it to finish, then open that version's
**Output** tab and download `model_export.zip`.

## 9. Convert to TF.js locally

Identical steps to the tomato model - reuse the same local scripts/venv from that run if you
still have them (`export_weights.py`, `build_keras2.py`, `convert.py` in your `model_export`
folder), just point them at this new `model.h5`. If starting fresh:

```powershell
python -m venv tfjs_env
tfjs_env\Scripts\activate
pip install tensorflowjs tf_keras
```

Then: load `model.h5` with plain Keras 3 and export its weights (`model.get_weights()` ->
`np.savez`), rebuild the identical architecture under `TF_USE_LEGACY_KERAS=1` (18-class Dense
output this time - read `NUM_CLASSES` from `metadata.json` rather than hardcoding it), load the
weights back in, save as H5, then run it through `tensorflowjs.converters.save_keras_model`. See
`CLAUDE.md`'s "Retraining the model" section for exactly why each of these steps is needed
(Keras 2 vs. 3 H5 format incompatibility, stale `tensorflow_decision_forests`/`tensorflow_hub`
imports inside `tensorflowjs`, deprecated `np.object` references).

## 10. Install into PlantDoctor

Copy `metadata.json` into the resulting `tfjs_model/` folder, then replace everything in
`public/model/` with those files (`model.json`, `group1-shard*.bin`, `metadata.json`). No app
code changes needed - `App.js` and `remedies.js` are both already generic over the label set.